# Schemas & Validation: Theories of What Matters

A **schema** is a structured template that defines:
- What fields exist
- What type each field should be
- What values are allowed

In humanities terms: **a schema is a theory of what matters about a text.**

Two scholars can use different schemas and both be legitimate. But once you pick a schema, you can compare texts consistently.

---

## 1. A Simple Schema

Let's define what a "poem profile" should look like:

In [ ]:
# ============================================================
# AVAILABLE COURSE READINGS (for reference)
# ============================================================
# "genesis", "ecclesiastes", "matthew", "hesiod", "homer_iliad"
# "aristotle_poetics", "aristotle_politics", "ovid_pygmalion"
# "keats_sonnets", "shakespeare_sonnets", "milton_paradise"
# "shelley_frankenstein", "shaw_pygmalion", "twain_eves_diary"
# ============================================================

# Our controlled vocabularies (enums)
ALLOWED_TONES = [
    "awe", "grief", "irony", "rage", "tenderness",
    "contemplation", "longing", "defiance", "wonder", "despair"
]

ALLOWED_DEVICES = [
    "metaphor", "simile", "alliteration", "anaphora", "enjambment",
    "repetition", "personification", "apostrophe", "imagery", "irony"
]

ALLOWED_STANCES = [
    "observer", "participant", "confessor", "prophet", "mourner",
    "lover", "critic", "witness", "dreamer"
]

ALLOWED_SENSES = ["sight", "sound", "touch", "taste", "smell"]

In [ ]:
# The schema as a description (what we expect)
POEM_SCHEMA = {
    "title": "string",
    "author": "string (optional)",
    "line_count": "int",
    "avg_line_length_words": "float",
    "tone": f"one of {ALLOWED_TONES}",
    "devices": f"list of items from {ALLOWED_DEVICES}",
    "speaker_stance": f"one of {ALLOWED_STANCES}",
    "dominant_sense": f"one of {ALLOWED_SENSES}",
    "imagery": "list of strings",
    "themes": "list of strings",
    "has_volta": "bool",
    "keywords": "list of strings"
}

print("Our poem schema:")
for field, field_type in POEM_SCHEMA.items():
    print(f"  {field}: {field_type}")

---

## 2. An Example Profile

Here's a profile that follows the schema:

In [ ]:
fog_profile = {
    "title": "Fog",
    "author": "Carl Sandburg",
    "line_count": 6,
    "avg_line_length_words": 3.5,
    "tone": "contemplation",
    "devices": ["metaphor", "personification", "imagery"],
    "speaker_stance": "observer",
    "dominant_sense": "sight",
    "imagery": ["fog", "cat", "harbor", "city", "haunches"],
    "themes": ["nature", "urban life", "transience"],
    "has_volta": False,
    "keywords": ["fog", "cat", "silent", "moves"]
}

fog_profile

---

## 3. Why Validation Matters

A profile might *look* right but break the rules. Validation catches mistakes before they cause problems downstream.

Here's a **bad** profile:

In [ ]:
bad_profile = {
    "title": "Fog",
    # missing author (okay, it's optional)
    "line_count": "six",           # Should be int, not string!
    "avg_line_length_words": 3.5,
    "tone": "vibes",                # Not in allowed tones!
    "devices": "metaphor",          # Should be a list!
    "speaker_stance": "observer",
    "dominant_sense": "sight",
    "imagery": ["fog", "cat"],
    "themes": ["nature"],
    "has_volta": "no",              # Should be bool, not string!
    "keywords": ["fog"]
}

---

## 4. Building a Validator

Let's write functions that check each rule:

In [ ]:
def validate_profile(profile):
    """Validate a poem profile against our schema. Returns list of errors."""
    errors = []
    
    # Required fields
    required_fields = [
        "title", "line_count", "avg_line_length_words", "tone",
        "devices", "speaker_stance", "dominant_sense", "imagery",
        "themes", "has_volta", "keywords"
    ]
    
    for field in required_fields:
        if field not in profile:
            errors.append(f"Missing required field: '{field}'")
    
    # Type checks
    if "title" in profile and not isinstance(profile["title"], str):
        errors.append(f"'title' must be a string, got {type(profile['title']).__name__}")
    
    if "line_count" in profile and not isinstance(profile["line_count"], int):
        errors.append(f"'line_count' must be an int, got {type(profile['line_count']).__name__}")
    
    if "avg_line_length_words" in profile and not isinstance(profile["avg_line_length_words"], (int, float)):
        errors.append(f"'avg_line_length_words' must be a number, got {type(profile['avg_line_length_words']).__name__}")
    
    if "has_volta" in profile and not isinstance(profile["has_volta"], bool):
        errors.append(f"'has_volta' must be a bool, got {type(profile['has_volta']).__name__}")
    
    # List checks
    list_fields = ["devices", "imagery", "themes", "keywords"]
    for field in list_fields:
        if field in profile and not isinstance(profile[field], list):
            errors.append(f"'{field}' must be a list, got {type(profile[field]).__name__}")
    
    # Enum checks
    if "tone" in profile and profile["tone"] not in ALLOWED_TONES:
        errors.append(f"'tone' must be one of {ALLOWED_TONES}, got '{profile['tone']}'")
    
    if "speaker_stance" in profile and profile["speaker_stance"] not in ALLOWED_STANCES:
        errors.append(f"'speaker_stance' must be one of {ALLOWED_STANCES}, got '{profile['speaker_stance']}'")
    
    if "dominant_sense" in profile and profile["dominant_sense"] not in ALLOWED_SENSES:
        errors.append(f"'dominant_sense' must be one of {ALLOWED_SENSES}, got '{profile['dominant_sense']}'")
    
    # Check devices list contains only allowed values
    if "devices" in profile and isinstance(profile["devices"], list):
        for device in profile["devices"]:
            if device not in ALLOWED_DEVICES:
                errors.append(f"Device '{device}' not in allowed list: {ALLOWED_DEVICES}")
    
    return errors

In [ ]:
# Validate the good profile
errors = validate_profile(fog_profile)
if errors:
    print("❌ Validation failed:")
    for e in errors:
        print(f"  - {e}")
else:
    print("✓ Profile is valid!")

In [ ]:
# Validate the bad profile
errors = validate_profile(bad_profile)
if errors:
    print("❌ Validation failed:")
    for e in errors:
        print(f"  - {e}")
else:
    print("✓ Profile is valid!")

---

## 5. The Interpretive Move

Notice what just happened:

1. We **chose** what fields matter (that's interpretation)
2. We **constrained** what values are allowed (that's theory)
3. We **enforced** those constraints (that's accountability)

A different scholar might:
- Add `meter` and `rhyme_scheme` fields
- Remove `dominant_sense`
- Use different tone categories

**Both schemas would be valid. They'd just notice different things.**

---

## 6. JSON: The Universal Format

When we ask an AI to produce structured output, we use **JSON** (JavaScript Object Notation).

JSON is just a text format for dictionaries and lists:

In [ ]:
import json

# Convert dict to JSON string
json_string = json.dumps(fog_profile, indent=2)
print("As JSON:")
print(json_string)

In [ ]:
# Convert JSON string back to dict
parsed = json.loads(json_string)
print(f"\nParsed back to dict: {type(parsed)}")
print(f"Title: {parsed['title']}")

This round-trip (`dict → JSON string → dict`) is how we'll communicate with AI models:

1. We describe the schema in the prompt
2. The model outputs JSON
3. We parse it back to a dict
4. We validate it against our rules

---

## ✏️ Exercise: Manual Extraction

Before we let AI do this, try it yourself. Fill in the profile for this poem:

In [ ]:
# From course readings: Shakespeare Sonnet 57
exercise_poem = """Sonnet 57
by William Shakespeare

Being your slave what should I do but tend,
Upon the hours, and times of your desire?
I have no precious time at all to spend;
Nor services to do, till you require.
Nor dare I chide the world-without-end hour,
Whilst I, my sovereign, watch the clock for you,
Nor think the bitterness of absence sour,
When you have bid your servant once adieu;
Nor dare I question with my jealous thought
Where you may be, or your affairs suppose,
But, like a sad slave, stay and think of nought
Save, where you are, how happy you make those.
So true a fool is love, that in your will,
Though you do anything, he thinks no ill."""

print(exercise_poem)

In [ ]:
# Fill this in!
my_profile = {
    "title": "Sonnet 57",
    "author": "William Shakespeare",
    "line_count": 14,
    "avg_line_length_words": 8.0,  # Estimate
    "tone": "longing",  # Pick from ALLOWED_TONES
    "devices": ["metaphor", "anaphora", "imagery"],  # Pick from ALLOWED_DEVICES
    "speaker_stance": "lover",  # Pick from ALLOWED_STANCES
    "dominant_sense": "sight",  # Pick from ALLOWED_SENSES
    "imagery": ["slave", "clock", "servant", "sovereign"],
    "themes": ["love", "devotion", "time"],
    "has_volta": True,  # The couplet is a turn
    "keywords": ["slave", "sovereign", "will", "fool", "love"]
}

In [ ]:
# Validate your profile
errors = validate_profile(my_profile)
if errors:
    print("❌ Fix these issues:")
    for e in errors:
        print(f"  - {e}")
else:
    print("✓ Your profile is valid!")
    print("\nYour profile:")
    print(json.dumps(my_profile, indent=2))

---

## The Category Theory Whisper 🔮

*A schema defines the **shape** of a transformation's output.*

When we say "extract a profile from a poem," we're defining a morphism:

```
extract: Poem (string) → Profile (dict matching schema)
```

The schema is a **contract**. It guarantees that no matter which poem goes in, the output will have the same structure.

This is what makes composition possible: if every profile has the same shape, we can write functions that work on *any* profile without knowing which poem it came from.

---

## Next: Agency and Action

Now you understand schemas and validation. Let's apply these ideas to a more complex analysis: tracking who speaks and who acts in a text.

→ Continue to `06_agency_and_action.ipynb`